# Four-environment dataset browser

Browse local Hugging Face downloads or collections under `logs`, without a model or simulator.
Edit the paths below, then run all cells. Paths may be absolute or relative to the repository root;
launching Jupyter from either the root or `analysis/` works. Zarr stores are opened read-only.
Only metadata is loaded for the overview; RGB is loaded for the selected episode.

In [ ]:
from pathlib import Path

# Set this explicitly if your notebook working directory is outside the repository.
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / 'datasets/common.py').is_file()), Path.cwd())

# Hugging Face: download and extract locally first (see README).
DATASETS = {
    'pusht': 'data/pusht/fresh_iid/manifest.json',
    'reacher': 'data/reacher/reacher_d_random_3k_v4.zarr',
    'beerpong': 'data/beerpong/beerpong_damping_9k_v2.zarr',
    'fetch_slide': 'data/fetch_slide/fetch_slide_friction_density_mass_xy_36k_v2.zarr',
}

# Alternatively, replace any individual path above with your own collection:
# DATASETS['pusht'] = 'logs/collection/pusht'  # directory containing manifest.json
# DATASETS['reacher'] = 'logs/collection/reacher.zarr'
# DATASETS['beerpong'] = 'logs/collection/beerpong.zarr'
# DATASETS['fetch_slide'] = 'logs/collection/fetch_slide.zarr'
# Use None to skip an environment.

In [ ]:
import io
import json
import sys
from html import escape
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image
from IPython.display import HTML, display

if str(REPO_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT.resolve()))
from datasets.pusht import PushTManifest
from datasets.reacher import ReacherEpisodes
from datasets.beerpong import BeerPongEpisodes
from datasets.fetch_slide import FetchSlideEpisodes

READERS = dict(pusht=PushTManifest, reacher=ReacherEpisodes,
               beerpong=BeerPongEpisodes, fetch_slide=FetchSlideEpisodes)
PHYSICS_NAMES = {
    'pusht': ['cog_x_px', 'cog_y_px'],
    'reacher': ['joint1_damping', 'joint2_damping'],
    'beerpong': ['contact_damping'],
    'fetch_slide': ['mass', 'friction', 'fluid_density'],
}
SPLITS = ('train', 'validation')
COLORS = ('tab:blue', 'tab:orange')

In [ ]:
def inspect_collection(env, path):
    path = Path(path).expanduser()
    if not path.is_absolute():
        path = REPO_ROOT / path
    if env == 'pusht' and path.is_dir():
        path = path / 'manifest.json'
    if not path.exists():
        raise FileNotFoundError(f'{path} — check the path or extract the dataset archive')
    collection = READERS[env](path)
    if not len(collection):
        raise ValueError(f'{path}: no episodes')
    lengths, splits, physics, dynamics, sigmas = [], [], [], [], []
    paths = [path.parent / shard['path'] for shard in collection.shards] if env == 'pusht' else [path]
    schema = None
    for shard_index, shard_path in enumerate(paths):
        group = collection.group(shard_path)
        data, meta = group['data'], group['meta']
        ends = meta['episode_ends'][:]
        if not len(ends):
            continue
        starts = np.r_[0, ends[:-1]]
        shard_lengths = ends - starts
        if np.any(shard_lengths <= 0):
            raise ValueError(f'{shard_path}: episode ends must strictly increase')
        shapes = {name: (data[name].shape[1:], str(data[name].dtype)) for name in ('img', 'state', 'action')}
        if any(len(data[name]) != int(ends[-1]) for name in shapes):
            raise ValueError(f'{shard_path}: image/state/action lengths differ from episode ends')
        if len(shapes['img'][0]) != 3 or shapes['img'][0][-1] != 3 or shapes['img'][1] != 'uint8':
            raise ValueError(f'{shard_path}: expected uint8 THWC RGB')
        if schema is not None and shapes != schema:
            raise ValueError(f'{shard_path}: shard shapes/dtypes differ')
        schema = shapes
        lengths.extend(shard_lengths.tolist())
        if env == 'pusht':
            shard = collection.shards[shard_index]
            if len(ends) != shard['episodes']:
                raise ValueError(f'{shard_path}: episode count differs from manifest')
            splits.extend([SPLITS.index(shard['split'])] * len(ends))
            # Stored CoG coordinates are normalized as [x/60, y/60 - 1].
            cog = data['cog_local_normed'].oindex[starts].astype(float)
            physics.extend((cog * 60 + [0, 60]).tolist())
            dynamics.extend(data['cog_seed'].oindex[starts].tolist())
            sigmas.extend(data['noise_sigma_px'].oindex[starts].tolist())
        else:
            splits.extend(collection.split.tolist())
            dynamics.extend(collection.dynamics.tolist())
            if env == 'beerpong':
                physics.extend(meta['episode_contact_damping'][:].reshape(-1, 1).tolist())
            else:
                table = collection.metadata['physics_params_by_dynamics']
                physics.extend([[table[int(i)][name] for name in PHYSICS_NAMES[env]]
                                for i in collection.dynamics])
    lengths, splits = np.asarray(lengths), np.asarray(splits)
    physics, dynamics = np.asarray(physics), np.asarray(dynamics)
    if any(len(values) != len(collection) for values in (lengths, splits, physics, dynamics)):
        raise ValueError(f'{path}: episode metadata lengths differ')
    if not np.isin(splits, [0, 1]).all() or not np.isfinite(physics).all():
        raise ValueError(f'{path}: invalid split labels or physics values')
    return dict(collection=collection, path=path, lengths=lengths, splits=splits,
                physics=physics, dynamics=dynamics, sigmas=np.asarray(sigmas), schema=schema)


def episode_details(env, info, episode_id):
    collection = info['collection']
    episode = collection.read_episode(episode_id)
    path, local_id, start, stop = collection.episode_info(episode_id)
    group = collection.group(path)
    details = dict(episode_id=int(episode_id), local_episode_id=int(local_id),
                   shard=str(path), frames=int(stop - start), split=SPLITS[info['splits'][episode_id]],
                   physics=dict(zip(PHYSICS_NAMES[env], info['physics'][episode_id].tolist())))
    for name, values in group['meta'].arrays():
        if name not in ('episode_ends', 'split') and len(values.shape) and len(values) == len(group['meta/episode_ends']):
            details[name] = np.asarray(values[local_id]).tolist()
    if env == 'pusht':
        for name in ('cog_seed', 'reset_seed', 'noise_sigma_px'):
            details[name] = np.asarray(group['data'][name][start]).tolist()
    elif env == 'beerpong':
        details['training_condition'] = np.asarray(group['meta/episode_action'][local_id]).tolist()
    elif env == 'fetch_slide':
        details['training_eligible'] = len(episode['action']) >= 20
        details['training_condition'] = (episode['action'][:2].reshape(-1).tolist()
                                         if len(episode['action']) >= 2 else None)
    return episode, details


def html_table(headers, rows):
    head = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    return HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>')

## Overview

Missing datasets are reported and skipped; malformed datasets raise an error.
Length statistics include all recorded episodes, including short FetchSlide episodes that training may filter out.
The representative image is the first recorded frame. No full image array is read here.

In [ ]:
collections = {}
for env, path in DATASETS.items():
    if path is None:
        continue
    try:
        collections[env] = inspect_collection(env, path)
    except FileNotFoundError as error:
        print(f'{env}: {error}')

rows = []
for env, info in collections.items():
    for label, split in enumerate(SPLITS):
        mask = info['splits'] == label
        lengths = info['lengths'][mask]
        stats = f'{lengths.min()} / {np.median(lengths):g} / {lengths.max()}' if len(lengths) else '—'
        rows.append([env, split, len(lengths), int(lengths.sum()), stats,
                     len(np.unique(info['dynamics'][mask]))])
display(html_table(['env', 'split', 'episodes', 'frames', 'length min / median / max', 'CoGs / dynamics'], rows))
display(html_table(['env', 'path', 'RGB shape / dtype', 'state shape / dtype', 'action shape / dtype'],
                   [[env, info['path'], *(f'{info["schema"][name][0]} / {info["schema"][name][1]}'
                     for name in ('img', 'state', 'action'))] for env, info in collections.items()]))

if collections:
    fig, axes = plt.subplots(1, len(collections), figsize=(4 * len(collections), 4), squeeze=False)
    for ax, (env, info) in zip(axes[0], collections.items()):
        path, _, start, _ = info['collection'].episode_info(0)
        ax.imshow(info['collection'].group(path)['data/img'][start])
        ax.set_title(env)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## Physics distributions

Physics plots count each CoG / dynamics ID once within each split, regardless of the number or length of episodes.
These are observed configuration distributions, not fitted probability densities.
Push-T CoG is in local pixels (it is not drawn in the recorded RGB).
BeerPong cup position is episode geometry and is plotted separately from damping.

In [ ]:
for env, info in collections.items():
    names = PHYSICS_NAMES[env]
    pairs = list(combinations(range(len(names)), 2))
    fig, axes = plt.subplots(1, len(names) + len(pairs), figsize=(4 * (len(names) + len(pairs)), 3.5), squeeze=False)
    for label, (split, color) in enumerate(zip(SPLITS, COLORS)):
        ids = np.flatnonzero(info['splits'] == label)
        _, unique = np.unique(info['dynamics'][ids], return_index=True)
        values = info['physics'][ids[unique]]
        for column, name in enumerate(names):
            bins = np.histogram_bin_edges(info['physics'][:, column], bins=15)
            axes[0, column].hist(values[:, column], bins=bins, alpha=.5, color=color, label=split)
            axes[0, column].set(xlabel=name, ylabel='CoGs / dynamics')
        for ax, (x, y) in zip(axes[0, len(names):], pairs):
            ax.scatter(values[:, x], values[:, y], s=18, alpha=.7, color=color, label=split)
            ax.set(xlabel=names[x], ylabel=names[y])
    axes[0, 0].legend()
    fig.suptitle(env)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    if env == 'pusht':
        display(html_table(['split', 'clean episodes (sigma = 0)', 'noisy episodes (sigma > 0)'],
                           [[split, int(np.sum((info['splits'] == label) & (info['sigmas'] == 0))),
                             int(np.sum((info['splits'] == label) & (info['sigmas'] > 0)))]
                            for label, split in enumerate(SPLITS)]))
    elif env == 'beerpong':
        fig, ax = plt.subplots(figsize=(5, 3))
        cups = info['collection'].root['meta/episode_cup_body_y'][:]
        bins = np.histogram_bin_edges(cups, bins=15)
        for label, (split, color) in enumerate(zip(SPLITS, COLORS)):
            ax.hist(cups[info['splits'] == label], bins=bins, alpha=.5, color=color, label=split)
        ax.set(xlabel='cup_body_y (m)', ylabel='episodes', title='BeerPong cup position')
        ax.legend()
        plt.show()
        plt.close(fig)

## Episode browser

Episode IDs are zero-based and global across Push-T shards. Select an ID from the chosen split, then click **Load episode**.
Only that episode's RGB/state/action arrays are materialized; the previous episode is released on loading another.
Playback interval is a display speed, not simulator time; the horizontal axis uses recorded frame indices.
Click trace names in the Plotly legend to hide/show coordinates. The red lines follow the current frame.

- **Push-T:** state = agent XY, block XY, block angle; action = absolute XY pixel target.
- **Reacher:** state = two joint angles and two velocities; action = two raw controls.
- **BeerPong:** state contains qpos followed by qvel (coordinate indices shown); action shows the recorded two-channel control.
  The episode-level training condition is shown in the metadata.
- **FetchSlide:** state is the raw robot observation; action is the recorded 4D control, including retraction.
  The training condition is the first two 4D hit rows flattened to 8D.

Trajectory panels show agent/block XY, joint-angle space, ball YZ, or puck XY, respectively.
The notebook does not write data or create output directories. Widgets require a frontend with ipywidgets support.

In [ ]:
def make_browser(collections):
    env_choice = widgets.Dropdown(options=list(collections), description='Env:')
    split_choice = widgets.Dropdown(options=SPLITS, description='Split:')
    episode_choice = widgets.BoundedIntText(value=0, min=0, max=0, description='Episode:')
    load_button = widgets.Button(description='Load episode')
    status = widgets.HTML()
    metadata = widgets.HTML(layout=widgets.Layout(max_width='850px'))
    image = widgets.Image(format='png', layout=widgets.Layout(width='320px'))
    frame = widgets.IntSlider(min=0, max=0, value=0, description='Frame:', continuous_update=True)
    play = widgets.Play(min=0, max=0, value=0, interval=100)
    interval = widgets.IntSlider(value=100, min=30, max=1000, step=10, description='ms/frame:')
    links = [widgets.jslink((play, 'value'), (frame, 'value')),
             widgets.jslink((interval, 'value'), (play, 'interval'))]
    charts = widgets.VBox()
    thumbnails = widgets.Output()
    current = {}

    def select_split(change=None):
        play.playing = False
        for chart in charts.children:
            chart.close()
        current.clear()
        image.value = b''
        charts.children = ()
        metadata.value = ''
        thumbnails.clear_output(wait=False)
        info = collections[env_choice.value]
        ids = np.flatnonzero(info['splits'] == SPLITS.index(split_choice.value))
        load_button.disabled = episode_choice.disabled = not len(ids)
        frame.disabled = play.disabled = True
        if len(ids):
            episode_choice.min = 0
            episode_choice.max = int(ids[-1])
            episode_choice.min = int(ids[0])
            episode_choice.value = int(ids[0])
            status.value = f'{len(ids)} episodes; ID range {ids[0]}–{ids[-1]} (some IDs may belong to the other split).'
        else:
            status.value = 'This split has no episodes.'

    def show_frame(change=None):
        if not current:
            return
        episode, figure = current['episode'], current['figure']
        index = frame.value
        buffer = io.BytesIO()
        Image.fromarray(episode['rgb'][index]).save(buffer, format='PNG')
        image.value = buffer.getvalue()
        with figure.batch_update():
            for line in figure.layout.shapes:
                line.x0 = line.x1 = index
        position = current['trajectory']
        current['trail'].data[-1].x = [position[index, 0]]
        current['trail'].data[-1].y = [position[index, 1]]

    def load_episode(button=None):
        env, episode_id = env_choice.value, episode_choice.value
        info = collections[env]
        if info['splits'][episode_id] != SPLITS.index(split_choice.value):
            status.value = 'This episode belongs to the other split. Choose another ID.'
            return
        play.playing = False
        for chart in charts.children:
            chart.close()
        current.clear()
        charts.children = ()
        try:
            episode, details = episode_details(env, info, episode_id)
        except (OSError, ValueError, KeyError, IndexError) as error:
            image.value = b''
            metadata.value = ''
            thumbnails.clear_output(wait=False)
            frame.disabled = play.disabled = True
            status.value = escape(f'Could not load episode: {error}')
            raise
        state, action = episode['state'], episode['action']
        figure = go.FigureWidget(make_subplots(rows=2, cols=1, shared_xaxes=True,
                                              subplot_titles=('Raw state', 'Recorded action')))
        labels = {'pusht': ['agent_x', 'agent_y', 'block_x', 'block_y', 'block_angle'],
                  'reacher': ['joint1_angle', 'joint2_angle', 'joint1_velocity', 'joint2_velocity']}.get(env)
        for row, (kind, values) in enumerate((('state', state), ('action', action)), 1):
            for column in range(values.shape[1]):
                name = labels[column] if kind == 'state' and labels else f'{kind}[{column}]'
                visible = True
                if kind == 'state' and env in ('beerpong', 'fetch_slide'):
                    visible = True if column in (range(7, 10) if env == 'beerpong' else range(3, 6)) else 'legendonly'
                figure.add_scatter(x=np.arange(len(values)), y=values[:, column], name=name,
                                   visible=visible, row=row, col=1)
            figure.add_vline(x=0, line_color='red', row=row, col=1)
        figure.update_layout(height=480, margin=dict(l=45, r=10, t=35, b=35))
        figure.update_xaxes(title_text='Recorded frame', row=2, col=1)
        if env == 'pusht':
            tracks, axes = [('agent', state[:, :2]), ('block', state[:, 2:4])], ('x (px)', 'y (px)')
        elif env == 'reacher':
            tracks, axes = [('joint angles', state[:, :2])], ('joint1 (rad)', 'joint2 (rad)')
        elif env == 'beerpong':
            tracks, axes = [('ball', state[:, 8:10])], ('ball y (m)', 'ball z (m)')
        else:
            tracks, axes = [('puck', state[:, 3:5])], ('puck x (m)', 'puck y (m)')
        trail = go.FigureWidget()
        for name, values in tracks:
            trail.add_scatter(x=values[:, 0], y=values[:, 1], mode='lines', name=name)
        position = tracks[-1][1]
        trail.add_scatter(x=[position[0, 0]], y=[position[0, 1]], mode='markers',
                          marker=dict(color='red', size=10), name='Current frame')
        trail.update_layout(height=320, xaxis_title=axes[0], yaxis_title=axes[1],
                            margin=dict(l=45, r=10, t=15, b=40))
        current.update(episode=episode, figure=figure, trail=trail, trajectory=position)
        frame.max = play.max = len(state) - 1
        frame.value = play.value = 0
        frame.disabled = play.disabled = False
        metadata.value = '<pre style="white-space:pre-wrap;overflow-wrap:anywhere">' + escape(json.dumps(details, indent=2)) + '</pre>'
        charts.children = (figure, trail)
        with thumbnails:
            thumbnails.clear_output(wait=True)
            indices = np.unique(np.linspace(0, len(state) - 1, min(8, len(state)), dtype=int))
            fig, axes = plt.subplots(1, len(indices), figsize=(2 * len(indices), 2.5), squeeze=False)
            for ax, index in zip(axes[0], indices):
                ax.imshow(episode['rgb'][index])
                ax.set_title(f'Frame {index}')
                ax.axis('off')
            fig.tight_layout()
            plt.show()
            plt.close(fig)
        status.value = f'Loaded {env} episode {episode_id}; {len(state)} frames.'
        show_frame()

    env_choice.observe(select_split, names='value')
    split_choice.observe(select_split, names='value')
    frame.observe(show_frame, names='value')
    load_button.on_click(load_episode)
    select_split()
    browser = widgets.VBox([widgets.HBox([env_choice, split_choice, episode_choice, load_button]),
                            status, widgets.HBox([play, frame, interval]),
                            widgets.HBox([image, metadata]), thumbnails, charts])
    # Keep Python-side widget links alive with the browser.
    browser._dataset_links = links
    return browser

if 'browser' in globals():
    for link in browser._dataset_links:
        link.unlink()
    old_widgets = [browser]
    while old_widgets:
        old_widget = old_widgets.pop()
        old_widgets.extend(getattr(old_widget, 'children', ()))
        old_widget.close()

if collections:
    browser = make_browser(collections)
    display(browser)
else:
    print('No datasets available. Edit DATASETS above or extract the downloaded archives, then rerun.')